# 📘 Module 3.1 — RNN Fundamentals

**Goal:** Understand Recurrent Neural Networks and why they're used for sequential data.

## Why RNNs for ADAS?
Driving involves **temporal sequences** — the current frame depends on what happened before:
- 🚗 **Trajectory Prediction:** Where will that car be in 3 seconds?
- 📊 **Sensor Fusion Over Time:** Combining radar/camera readings over time
- 🛣️ **Driver Behavior Modeling:** Is the driver drowsy? Distracted?
- 🔄 **Video Understanding:** Processing video frame sequences

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. The Problem with Feedforward Networks

Feedforward networks (MLPs, CNNs) process each input **independently**. They have no memory of previous inputs.

```
Feedforward:  Input → Network → Output  (no memory)
Recurrent:    Input + Previous State → Network → Output + New State  (has memory!)
```

### RNN Cell — The Basic Unit

```
            ┌──────────┐
  x_t  ───►│          │───► h_t (output/hidden state)
            │  RNN     │
  h_{t-1}─►│  Cell    │───► h_t (passed to next step)
            └──────────┘
            
h_t = tanh(W_xh · x_t + W_hh · h_{t-1} + b)
```

In [ ]:
# --- Manual RNN Cell ---
# Let's implement an RNN cell from scratch to understand it

class ManualRNNCell:
    """Simple RNN cell implemented from scratch."""
    def __init__(self, input_size, hidden_size):
        # Initialize weights (normally these are learned)
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.1
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.b = np.zeros(hidden_size)
        self.hidden_size = hidden_size
    
    def forward(self, x_t, h_prev):
        """Process one time step."""
        # h_t = tanh(W_xh @ x_t + W_hh @ h_prev + b)
        h_t = np.tanh(self.W_xh @ x_t + self.W_hh @ h_prev + self.b)
        return h_t

# Create RNN cell: 4 input features, 8 hidden units
rnn_cell = ManualRNNCell(input_size=4, hidden_size=8)

# Process a sequence of 5 time steps
sequence = np.random.randn(5, 4)  # 5 steps, 4 features each
h = np.zeros(8)  # Initial hidden state

print("Processing sequence through RNN:")
for t in range(len(sequence)):
    h = rnn_cell.forward(sequence[t], h)
    print(f"  Step {t}: input shape={sequence[t].shape}, hidden state norm={np.linalg.norm(h):.4f}")

print(f"\nFinal hidden state encodes the ENTIRE sequence information!")

## 2. PyTorch RNN

In [ ]:
# --- PyTorch RNN ---
# Input:  (batch_size, sequence_length, input_size)
# Output: (batch_size, sequence_length, hidden_size)

rnn = nn.RNN(
    input_size=4,       # Features per time step
    hidden_size=8,      # Hidden state size
    num_layers=1,       # Number of stacked RNN layers
    batch_first=True,   # Input shape: (batch, seq, features)
)

# Simulated sensor data: 16 sequences, 20 time steps, 4 sensor readings
x = torch.randn(16, 20, 4)
h0 = torch.zeros(1, 16, 8)  # Initial hidden state

output, h_final = rnn(x, h0)

print(f"Input shape:        {x.shape}      (batch, seq_len, features)")
print(f"Output shape:       {output.shape}  (batch, seq_len, hidden)")
print(f"Final hidden shape: {h_final.shape}    (layers, batch, hidden)")
print(f"\nOutput at last step == final hidden: {torch.allclose(output[:, -1, :], h_final[0])}")

## 3. The Vanishing Gradient Problem

**Problem:** Standard RNNs struggle with long sequences because gradients become tiny (vanish) or huge (explode) during backpropagation through time.

```
Step 1 ← Step 2 ← Step 3 ← ... ← Step 100
                    Gradients get multiplied many times
                    → Vanish (< 1) or Explode (> 1)
```

**Solution:** LSTM and GRU (next section!)

In [ ]:
# --- Demonstrating the vanishing gradient ---

# Simulate gradient flow through many time steps
seq_lengths = [10, 50, 100, 200]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for seq_len in seq_lengths:
    # RNN gradient roughly = product of derivatives across time steps
    # tanh derivative is at most 1.0, typically < 1
    gradient_norms = [1.0]
    for t in range(1, seq_len):
        # Simulate gradient multiplied by random Jacobian
        grad_scale = np.random.uniform(0.7, 1.0)
        gradient_norms.append(gradient_norms[-1] * grad_scale)
    
    axes[0].plot(gradient_norms, label=f'Length={seq_len}')
    axes[1].semilogy(gradient_norms, label=f'Length={seq_len}')

axes[0].set_title('Gradient Magnitude (Linear Scale)')
axes[0].set_xlabel('Time Step (backward)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Gradient Magnitude (Log Scale)')
axes[1].set_xlabel('Time Step (backward)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Vanishing Gradients in Standard RNNs', fontsize=14)
plt.tight_layout()
plt.show()
print("→ Gradients vanish exponentially! LSTM/GRU solve this.")

---
## ✅ Key Takeaways

1. **RNNs** process sequences by maintaining a hidden state across time steps
2. The **hidden state** encodes information about all previous inputs
3. **Vanilla RNNs** suffer from vanishing gradients on long sequences
4. In ADAS, sequential data includes sensor readings, video frames, and trajectories

---
## 📖 Next Steps
➡️ **Next notebook:** [02_lstm_gru_sequence_modeling.ipynb](02_lstm_gru_sequence_modeling.ipynb) — LSTM & GRU solve the vanishing gradient problem